# Network GEC Runner

Parameterized Cost-Weighted GEC classification notebook for any network combination.
Edit the `EXPERIMENT` dict below, run GAAE runner first to get the encoder checkpoint,
then run all cells here.

| `network_combo` | `data_version` | `n_rois` | `file_suffix` |
|---|---|---|---|
| `dmn` | `__v4__` | 46 | `_dmn_correlation_matrix_z_transformed.npz` |
| `hippo` | `__v5__` | 4 | `_hippocampus_correlation_matrix_z_transformed.npz` |
| `limbic` | `__v6__` | 12 | `_limbic_correlation_matrix_z_transformed.npz` |
| `dan` | `__v7__` | 26 | `_dorsal_attention_correlation_matrix_z_transformed.npz` |
| `dmn_hippo` | `__v8__` | 50 | `_dmn_hippo_correlation_matrix_z_transformed.npz` |
| `dmn_limbic` | `__v9__` | 58 | `_dmn_limbic_correlation_matrix_z_transformed.npz` |
| `dmn_limbic_hippo` | `__v10__` | 62 | `_dmn_limbic_hippo_correlation_matrix_z_transformed.npz` |
| `all_combined` | `__v11__` | 88 | `_all_combined_correlation_matrix_z_transformed.npz` |

In [ ]:
# ── EXPERIMENT CONFIG ─────────────────────────────────────────────────────────
EXPERIMENT = {
    "network_combo": "dmn_hippo",
    "data_version": "__v8__",
    "n_rois": 50,
    "file_suffix": "_dmn_hippo_correlation_matrix_z_transformed.npz",
    "knn_k": 8,
    "use_pretrained_encoder": True,   # False = end-to-end GEC without GAAE pretraining
}
# ─────────────────────────────────────────────────────────────────────────────

REPO_ROOT = '/mnt/e/fyassine/ad-early-detection'
CLASSIFIER_ROOT = f'{REPO_ROOT}/CLASSIFIER'

WB_ROOT = f"{REPO_ROOT}/DATA/DELCODE/{EXPERIMENT['data_version']}/matrices"
METADATA_DIR = f"{REPO_ROOT}/DATA/DELCODE/__v3__/metadata"
COHORTS_CSV = f'{METADATA_DIR}/cohorts.csv'
GEC_SPLITS_DIR = f'{METADATA_DIR}/splits_gec'
TRAIN_SPLIT_CSV = f'{GEC_SPLITS_DIR}/train.csv'
VAL_SPLIT_CSV = f'{GEC_SPLITS_DIR}/val.csv'

GAAE_CHECKPOINT_SEARCH_DIR = f"{CLASSIFIER_ROOT}/notebooks/checkpoints_gaae_{EXPERIMENT['network_combo']}"
GEC_OUTPUT_DIR = f"{CLASSIFIER_ROOT}/notebooks/checkpoints_gec_{EXPERIMENT['network_combo']}"
WANDB_PROJECT = f"gec-network-{EXPERIMENT['network_combo']}"

# Model hyperparams (overridden from GAAE run_config.json if pretrained encoder is used)
IN_FEATURES = EXPERIMENT["n_rois"]
HIDDEN_DIM = max(IN_FEATURES, 64)
LATENT_DIM = 64
NUM_HEADS = 2
COND_DIM = 2
DROPOUT = 0.3
CLASSIFIER_HIDDEN = 32

# Training
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 25
EARLY_STOPPING_PATIENCE = 30
USE_SCHEDULER = True
USE_CLASS_COST_WEIGHTS = True
NORMALIZE_CLASS_COST_WEIGHTS = True
FREEZE_ENCODER = False

N_FOLDS = 5
RANDOM_STATE = 42

print(f"Experiment: {EXPERIMENT['network_combo']}")
print(f"Data root: {WB_ROOT}")
print(f"Pretrained encoder: {EXPERIMENT['use_pretrained_encoder']}")
print(f"IN_FEATURES: {IN_FEATURES}")

## Imports

In [ ]:
import json
import os
import sys
import tempfile
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import wandb
import torch
from matplotlib import pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from torch_geometric.loader import DataLoader

classifier_root = Path(CLASSIFIER_ROOT)
if str(classifier_root) not in sys.path:
    sys.path.insert(0, str(classifier_root))

from common.dataset import ClassificationDataset, CombinedClassificationDataset
from model.CostWeightedGEC.models import GraphEncoderClassifierAttention
from common.utils import (
    load_frozen_encoder_from_gaae,
    compute_class_weights,
    compute_class_cost_weights,
    knn_binary_adjacency_matrix_no_diag,
    set_seed,
)
from model.CostWeightedGEC.train import train_classifier
from common.validation import run_kfold_cv

sns.set_theme(style='whitegrid')
print('Imports OK')

## Setup

In [ ]:
try:
    wandb.login()
except Exception:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

set_seed(RANDOM_STATE)
os.makedirs(GEC_OUTPUT_DIR, exist_ok=True)

## GAAE Encoder Checkpoint

In [ ]:
GAAE_CHECKPOINT_PATH = None

if EXPERIMENT['use_pretrained_encoder']:
    checkpoint_candidates = sorted(
        [
            (run_dir.name, str(model_file), str(run_dir))
            for base_dir in [Path(GAAE_CHECKPOINT_SEARCH_DIR)]
            if base_dir.is_dir()
            for run_dir in sorted(base_dir.iterdir())
            if run_dir.is_dir()
            for model_file in [run_dir / f'model_{run_dir.name}.pth']
            if model_file.exists()
        ],
        key=lambda x: x[0],
    )

    if not checkpoint_candidates:
        raise FileNotFoundError(
            f'No GAAE checkpoints found in {GAAE_CHECKPOINT_SEARCH_DIR}.\n'
            f'Run NETWORK_GAAE_RUNNER.ipynb with network_combo="{EXPERIMENT["network_combo"]}" first.'
        )

    print('Available GAAE checkpoints:')
    for i, (run_name, _, run_dir) in enumerate(checkpoint_candidates):
        print(f'  {i}: {run_name}')

    default_index = len(checkpoint_candidates) - 1
    selected_text = input(f'Select checkpoint index [default {default_index}]: ').strip()
    selected_index = default_index if selected_text == '' else int(selected_text)

    RUN_NAME, GAAE_CHECKPOINT_PATH, SELECTED_RUN_DIR = checkpoint_candidates[selected_index]

    # Load model dimensions from run_config.json if available
    run_config_path = Path(SELECTED_RUN_DIR) / 'run_config.json'
    if run_config_path.exists():
        with open(run_config_path, 'r') as f:
            run_config = json.load(f)
        mc = run_config.get('model_config', {})
        IN_FEATURES = int(mc.get('in_features', IN_FEATURES))
        HIDDEN_DIM = int(mc.get('hidden_size', HIDDEN_DIM))
        LATENT_DIM = int(mc.get('latent_dim', LATENT_DIM))
        NUM_HEADS = int(mc.get('attention_heads', NUM_HEADS))
        DROPOUT = float(mc.get('dropout', DROPOUT))
        print(f'Loaded dims from run_config: in={IN_FEATURES}, hidden={HIDDEN_DIM}, latent={LATENT_DIM}')

    print(f'Using GAAE checkpoint: {GAAE_CHECKPOINT_PATH}')
else:
    print('Pretrained encoder disabled — training GEC end-to-end.')

## Load Datasets

In [ ]:
adjacency_args = {'k': EXPERIMENT['knn_k']}
allowed_diagnoses = {'converter', 'mci'}

cohorts_df = pd.read_csv(COHORTS_CSV)
cohort_id_col = next(
    (c for c in ['Repseudonym', 'Pseudonym', 'ID'] if c in cohorts_df.columns), None
)
if cohort_id_col is None:
    raise ValueError(f'Missing subject ID column in cohorts CSV: {list(cohorts_df.columns)}')
cohorts_df[cohort_id_col] = cohorts_df[cohort_id_col].astype(str)
cohorts_df['diagnosis'] = cohorts_df['diagnosis'].astype(str).str.lower().str.strip()

split_paths = {'train': TRAIN_SPLIT_CSV, 'val': VAL_SPLIT_CSV}
split_frames = {}
base_required_cols = {'Repseudonym', 'sex', 'age', 'diagnosis'}

for name, path in split_paths.items():
    df = pd.read_csv(path)
    missing = base_required_cols - set(df.columns)
    assert not missing, f'Split {name}.csv missing columns: {missing}'
    df = df.copy()
    df['Repseudonym'] = df['Repseudonym'].astype(str)
    df['diagnosis'] = df['diagnosis'].astype(str).str.lower().str.strip()
    df['converter_status'] = (df['diagnosis'] == 'converter').astype(int)
    split_frames[name] = df
    print(f'{name}: subjects={df["Repseudonym"].nunique()}, counts={df["diagnosis"].value_counts().to_dict()}')

all_splits_df = pd.concat(split_frames.values(), ignore_index=True).drop_duplicates(subset=['Repseudonym'])
all_splits_df['Repseudonym'] = all_splits_df['Repseudonym'].astype(str)

patient_info_path = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False).name
all_splits_df[['Repseudonym', 'sex', 'age']].rename(columns={'Repseudonym': 'Pseudonym'}).set_index('Pseudonym').to_csv(patient_info_path)

converter_ids = sorted(all_splits_df.loc[all_splits_df['diagnosis'] == 'converter', 'Repseudonym'].unique())
non_converter_ids = sorted(all_splits_df.loc[all_splits_df['diagnosis'] == 'mci', 'Repseudonym'].unique())

converter_dataset = ClassificationDataset(
    root=WB_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    patient_info_path=patient_info_path,
    is_converter_dataset=True,
    separator=',',
    subject_ids=converter_ids,
    file_suffix=EXPERIMENT['file_suffix'],
    filter_csv_path=None,
    converter_list_path=None,
)

non_converter_dataset = ClassificationDataset(
    root=WB_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    patient_info_path=patient_info_path,
    is_converter_dataset=False,
    separator=',',
    subject_ids=non_converter_ids,
    file_suffix=EXPERIMENT['file_suffix'],
    filter_csv_path=None,
    converter_list_path=None,
)

print(f'Converter dataset: {len(converter_dataset)} scans, {len(converter_ids)} subjects')
print(f'Non-converter dataset: {len(non_converter_dataset)} scans, {len(non_converter_ids)} subjects')

## Load Splits

In [ ]:
combined_dataset = CombinedClassificationDataset(converter_dataset, non_converter_dataset)
all_labels = combined_dataset.get_labels()

patient_to_indices = {}
index_to_patient = {}
for idx in range(len(combined_dataset)):
    patient_id = str(getattr(combined_dataset[idx], 'patient_id', ''))
    index_to_patient[idx] = patient_id
    patient_to_indices.setdefault(patient_id, []).append(idx)

train_ids = set(split_frames['train']['Repseudonym'].astype(str))
val_ids = set(split_frames['val']['Repseudonym'].astype(str))

def get_split_indices(patient_ids):
    indices = []
    for pid in patient_ids:
        indices.extend(patient_to_indices.get(str(pid), []))
    return sorted(indices)

train_split_indices = get_split_indices(train_ids)
val_split_indices = get_split_indices(val_ids)
cv_indices = train_split_indices + val_split_indices
cv_labels = [all_labels[i] for i in cv_indices]
cv_patient_ids = [index_to_patient[i] for i in cv_indices]

print(f'Train scans: {len(train_split_indices)}, Val scans: {len(val_split_indices)}')
print(f'CV pool: {len(cv_indices)} scans, {len(set(cv_patient_ids))} subjects')
print(f'Converter scan rate in CV: {sum(cv_labels)/len(cv_labels)*100:.1f}%')

## 5-Fold Stratified Cross-Validation

In [ ]:
config = {
    'N_FOLDS': N_FOLDS,
    'RANDOM_STATE': RANDOM_STATE,
    'BATCH_SIZE': BATCH_SIZE,
    'USE_CLASS_COST_WEIGHTS': USE_CLASS_COST_WEIGHTS,
    'NORMALIZE_CLASS_COST_WEIGHTS': NORMALIZE_CLASS_COST_WEIGHTS,
    'GAAE_CHECKPOINT_PATH': GAAE_CHECKPOINT_PATH,
    'FREEZE_ENCODER': FREEZE_ENCODER,
    'LEARNING_RATE': LEARNING_RATE,
    'EPOCHS': EPOCHS,
    'EARLY_STOPPING_PATIENCE': EARLY_STOPPING_PATIENCE,
    'WANDB_PROJECT': WANDB_PROJECT,
    'USE_SCHEDULER': USE_SCHEDULER,
}

model_kwargs = {
    'in_features': IN_FEATURES,
    'hidden_dim': HIDDEN_DIM,
    'latent_dim': LATENT_DIM,
    'cond_dim': COND_DIM,
    'num_heads': NUM_HEADS,
    'dropout': DROPOUT,
    'classifier_hidden': CLASSIFIER_HIDDEN,
}

(
    cv_results, cv_histories, best_model_state, best_model,
    best_val_auc, best_fold, best_threshold_overall, best_f1_threshold,
    oof_preds, oof_targets,
) = run_kfold_cv(
    model_class=GraphEncoderClassifierAttention,
    model_kwargs=model_kwargs,
    combined_dataset=combined_dataset,
    cv_indices=cv_indices,
    cv_labels=cv_labels,
    cv_patient_ids=cv_patient_ids,
    index_to_patient=index_to_patient,
    all_labels=all_labels,
    config=config,
    device=device,
)

## Threshold Method Selection

In [ ]:
oof_preds_arr = np.array(oof_preds)
oof_targets_arr = np.array(oof_targets)

def _oof_metrics(thr):
    preds = (oof_preds_arr >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(oof_targets_arr, preds).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1   = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    return sens, spec, f1

y_sens, y_spec, y_f1 = _oof_metrics(best_threshold_overall)
f_sens, f_spec, f_f1 = _oof_metrics(best_f1_threshold)

print('OOF threshold options:')
print(f'  [1] Youden\'s J   thr={best_threshold_overall:.4f}  '
      f'sens={y_sens:.3f}  spec={y_spec:.3f}  F1={y_f1:.3f}')
print(f'  [2] Best F1      thr={best_f1_threshold:.4f}  '
      f'sens={f_sens:.3f}  spec={f_spec:.3f}  F1={f_f1:.3f}')
print()

threshold_choice = input('Select threshold method [1=Youden (default), 2=Best F1]: ').strip()

if threshold_choice == '2':
    ACTIVE_THRESHOLD = float(best_f1_threshold)
    THRESHOLD_METHOD = 'oof_f1'
    print(f'Using F1-optimal threshold: {ACTIVE_THRESHOLD:.4f}')
else:
    ACTIVE_THRESHOLD = float(best_threshold_overall)
    THRESHOLD_METHOD = 'oof_youden'
    print(f'Using Youden threshold: {ACTIVE_THRESHOLD:.4f}')

## Cross-Validation Results

In [ ]:
print(f'\nExperiment: {EXPERIMENT["network_combo"]} ({EXPERIMENT["n_rois"]} ROIs)')
print(f'Active threshold: {ACTIVE_THRESHOLD:.4f} ({THRESHOLD_METHOD})')
print('Cross-Validation Summary:')
print('=' * 60)
print(f'{"Metric":<20} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
print('-' * 60)
for metric in ['val_auc', 'val_sensitivity', 'val_specificity', 'val_f1']:
    values = cv_results[metric]
    print(f'{metric:<20} {np.mean(values):>10.4f} {np.std(values):>10.4f} {np.min(values):>10.4f} {np.max(values):>10.4f}')
print(f'\nBest Fold: {best_fold} (AUC={best_val_auc:.4f})')
print(f'OOF Youden threshold: {best_threshold_overall:.4f}')
print(f'OOF F1-optimal threshold: {best_f1_threshold:.4f}')
print(f'>>> Active threshold: {ACTIVE_THRESHOLD:.4f} ({THRESHOLD_METHOD}) <<<')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

metrics = ['val_auc', 'val_sensitivity', 'val_specificity', 'val_f1']
titles = ['AUC-ROC', 'Sensitivity', 'Specificity', 'F1-Score']
colors = sns.color_palette('colorblind', len(metrics))

for ax, metric, title, color in zip(axes.flat, metrics, titles, colors):
    values = np.array(cv_results[metric], dtype=float)
    fold_x = np.arange(1, len(values) + 1)
    mean_val = float(np.mean(values))
    std_val = float(np.std(values))
    bars = ax.bar(fold_x, values, color=color, alpha=0.82, edgecolor='black', linewidth=0.4)
    ax.plot(fold_x, values, color='black', marker='o', linewidth=1.2, alpha=0.75)
    ax.axhline(y=mean_val, color='crimson', linestyle='--', linewidth=1.6,
               label=f'Mean: {mean_val:.3f} +/- {std_val:.3f}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks(fold_x)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', linestyle=':', alpha=0.45)
    ax.legend(loc='lower right', fontsize=9)
    for rect, val in zip(bars, values):
        ax.text(rect.get_x() + rect.get_width() / 2.0, val + 0.02, f'{val:.2f}',
                ha='center', va='bottom', fontsize=8)

encoder_label = 'Pretrained GAAE encoder' if EXPERIMENT['use_pretrained_encoder'] else 'End-to-end'
fig.suptitle(
    f'Cost-Weighted GEC | {EXPERIMENT["network_combo"]} ({EXPERIMENT["n_rois"]} ROIs) | {encoder_label}\n'
    f'5-Fold Stratified CV | Threshold: {ACTIVE_THRESHOLD:.4f} ({THRESHOLD_METHOD})',
    fontsize=13,
)
plt.show()

## Save Results

In [ ]:
if best_model_state is None:
    raise RuntimeError('No model state to save. Run cross-validation first.')

run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
try:
    wandb_run_name = wandb.run.name if wandb and wandb.run and wandb.run.name else 'cv_run'
except Exception:
    wandb_run_name = 'cv_run'
run_name = f'{wandb_run_name}_{run_timestamp}'

run_artifact_dir = os.path.join(GEC_OUTPUT_DIR, run_name)
os.makedirs(run_artifact_dir, exist_ok=True)

model_file = os.path.join(run_artifact_dir, f'best_model_fold{best_fold}.pth')
torch.save(best_model_state, model_file)
print(f'Saved model: {model_file}')

run_summary = {
    'run_name': run_name,
    'timestamp': run_timestamp,
    'experiment': EXPERIMENT,
    'n_folds': N_FOLDS,
    'best_fold': int(best_fold),
    'best_val_auc': float(best_val_auc),
    'best_threshold': float(ACTIVE_THRESHOLD),
    'threshold_method': THRESHOLD_METHOD,
    'youden_threshold': float(best_threshold_overall),
    'f1_threshold': float(best_f1_threshold),
    'cv_results': cv_results,
    'cv_histories': {
        'train_loss': cv_histories.get('train_loss', []),
        'val_loss': cv_histories.get('val_loss', []),
    },
    'config': {
        'in_features': IN_FEATURES,
        'hidden_dim': HIDDEN_DIM,
        'latent_dim': LATENT_DIM,
        'cond_dim': COND_DIM,
        'num_heads': NUM_HEADS,
        'dropout': DROPOUT,
        'classifier_hidden': CLASSIFIER_HIDDEN,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'knn_k': EXPERIMENT['knn_k'],
        'file_suffix': EXPERIMENT['file_suffix'],
        'freeze_encoder': FREEZE_ENCODER,
        'gaae_checkpoint': GAAE_CHECKPOINT_PATH,
        'wb_root': WB_ROOT,
        'use_scheduler': USE_SCHEDULER,
        'use_class_cost_weights': USE_CLASS_COST_WEIGHTS,
        'normalize_class_cost_weights': NORMALIZE_CLASS_COST_WEIGHTS,
        'use_pretrained_encoder': EXPERIMENT['use_pretrained_encoder'],
    },
}
summary_file = os.path.join(run_artifact_dir, 'run_summary.json')
with open(summary_file, 'w') as f:
    json.dump(run_summary, f, indent=2, default=str)
print(f'Saved run summary: {summary_file}')
print(f'Active threshold ({THRESHOLD_METHOD}): {ACTIVE_THRESHOLD:.4f}')

try:
    wandb.finish()
except Exception:
    pass
print(f'Done. Artifact dir: {run_artifact_dir}')